# Chapter 4: Time-Series Data in Pandas 🧠

A beginner-friendly workbook packed with detailed explanations, visual examples, and practice exercises:
1. **4.1 Creating Time Series**: Parsing dates with `pd.to_datetime()`, custom `format` codes, combining date columns, and generating sequences with `pd.date_range()`
2. **4.2 Slicing, Timedelta Math & Resampling**: Partial date slicing with `.loc[]`, calculating elapsed time with `Timedelta`, and changing frequencies with `.resample()`

---
---

## 4.1 Creating Time Series (`to_datetime` & `date_range`)

### Concept Summary
**Time-series data** is simply data where every row is stamped with a specific date or time (such as hourly weather temperatures, daily stock prices, or website clicks per minute).

In Python and Pandas, real **datetime objects** store individual time pieces (`year`, `month`, `day`, `hour`, `minute`, `second`, `microsecond`, and optional `tzinfo` for time zones). Converting plain text dates into real datetime objects unlocks smart calendar math, automatic weekday detection, and easy date slicing.

---

### Tools for Creating Time Series: Deep Dive with Examples

#### 1. Converting Strings to Dates (`pd.to_datetime()`)
Pandas automatically recognizes most common date styles without any extra instructions:
- **Standard Dates:** `'2024-08-05'`, `'05/08/2024'`, `'08/05/2024'`
- **Date + Time:** `'2024-08-05 14:30:00'`
- **Month Names:** `'05-Aug-2024'`, `'August 5, 2024'`
- **No Separators:** `'20240805'` (read as `YYYYMMDD`)

```python
d = pd.to_datetime('2024-08-05')
print(d)         # Output: 2024-08-05 00:00:00
print(d.year)    # Output: 2024
print(d.month)   # Output: 8
```

#### 2. Handling Unusual Date Formats (`format=...`)
If your text contains weird symbols (like `@`) or unusual ordering, tell Pandas how to read it using the `format` parameter and format codes:

| Code | Meaning | Example |
| :---: | :--- | :--- |
| `%Y` | 4-digit year | `2024` |
| `%y` | 2-digit year | `24` |
| `%m` | 2-digit month (`01`–`12`) | `08` |
| `%B` | Full month name | `August` | 
| `%b` | Short month name | `Aug` |
| `%d` | 2-digit day (`01`–`31`) | `05` |
| `%H` | Hour in 24-hour clock (`00`–`23`) | `14` |
| `%M` | Minute (`00`–`59`) | `30` |
| `%S` | Second (`00`–`59`) | `00` |

**Example (Parsing a custom string with `@`):**
```python
date_str = '2024 August 05 @ 14:30:00'
clean_date = pd.to_datetime(date_str, format='%Y %B %d @ %H:%M:%S')
print(clean_date)  # Output: 2024-08-05 14:30:00
```

#### 3. Combining Separate Year, Month, Day Columns
If a DataFrame stores `'year'`, `'month'`, and `'day'` in separate columns, pass all three into `pd.to_datetime()` to build a single date index:
```python
df = pd.DataFrame({
    'year':  [2024, 2025, 2026],
    'month': [8, 9, 10],
    'day':   [5, 15, 25],
    'vals':  [10, 20, 30]
})

# Main Step: 
# Combine into one datetime column, make it the index, and remove old columns
df['date'] = pd.to_datetime(df[['year', 'month', 'day']])
# Output:
#     year  month  day  vals       date
# 0  2024      8    5    10   2024-08-05
# 1  2025      9   15    20   2025-09-15
# 2  2026     10   25    30   2026-10-25

# Extra steps: 
# Set the new date column as the index and drop the old columns
df.set_index('date', inplace=True) # Set the new date column as the index

df.drop(columns=['year', 'month', 'day'], inplace=True)
# Output:
#             vals
# date            
# 2024-08-05    10
# 2025-09-15    20
# 2026-10-25    30  
```

#### 4. Generating Time Sequences (`pd.date_range()`)
`pd.date_range(start_date, end_date, periods, freq)` builds an evenly spaced sequence of timestamps (`DatetimeIndex`) automatically:
- **Method A (`start` + `end` + `freq`):** Steps from start to end at a fixed interval (e.g., `'D'` = daily, `'2h'` = every 2 hours, `'ME'` = month end).
  ```python
  # Every 2 hours from July 21 to July 22
  dr = pd.date_range(start='2024-07-21', end='2024-07-22', freq='2h')
  print(dr)
  # Output:
  # DatetimeIndex(['2024-07-21 00:00:00',
  #                '2024-07-21 02:00:00',
  #                '2024-07-21 04:00:00',
  #                '2024-07-21 06:00:00',
  #                '2024-07-21 08:00:00',
  #                '2024-07-21 10:00:00',
  #                '2024-07-21 12:00:00',
  #                '2024-07-21 14:00:00',
  #                '2024-07-21 16:00:00',
  #                '2024-07-21 18:00:00',
  #                '2024-07-21 20:00:00',
  #                '2024-07-21 22:00:00',
  #                '2024-07-22 00:00:00'],
  #               dtype='datetime64[ns]', freq='2H')
  ```
- **Method B (`start` + `periods`):** Generates a fixed count of timestamps starting from `start`.
  ```python
  # 5 daily dates starting Jan 1, 2024
  dr = pd.date_range(start='2024-01-01', periods=5, freq='D')
  print(dr)
  # Output:
  # DatetimeIndex(['2024-01-01', '2024-01-02', '2024-01-03', '2024-01-04', '2024-01-05'], dtype='datetime64[ns]', freq='D') 
  ```
- **Method C (`start` + `end` + `periods`):** Splits the exact span between `start` and `end` into `periods` equally spaced points.

In [11]:
# --- Demo: Creating Time-Series DataFrames ---
import pandas as pd
import numpy as np


# 1. Parsing a custom date string using format codes
print("----- Demo Question 1 -----")

raw_stamp = '2024 August 05 @ 14:30:00'
parsed = pd.to_datetime(raw_stamp, format='%Y %B %d @ %H:%M:%S')
print("Parsed Timestamp:", parsed, "| Year:", parsed.year)



# 2. Combining separate columns into a DatetimeIndex
print("\n\n----- Demo Question 2 -----")

df_parts = pd.DataFrame({
    'year':  [2024, 2025, 2026],
    'month': [8, 9, 10],
    'day':   [5, 15, 25],
    'vals':  [1, 2, 3]
})
df_parts['date'] = pd.to_datetime(df_parts[['year', 'month', 'day']])
df_parts.set_index('date', inplace=True)
df_parts.drop(columns=['year', 'month', 'day'], inplace=True)

print("\n=== DataFrame Indexed by Date ===")
print(df_parts)



# 3. Generating a synthetic time series with pd.date_range
print("\n\n----- Demo Question 3 -----")

dr = pd.date_range(start='1999-07-21', end='1999-09-21', freq='2h')
ts_df = pd.DataFrame({'data': np.random.randint(0, 100, len(dr))}, index=dr)

print("\n=== Generated 2-Hourly Time Series (First 5 Rows) ===")
print(ts_df.head())

----- Demo Question 1 -----
Parsed Timestamp: 2024-08-05 14:30:00 | Year: 2024


----- Demo Question 2 -----

=== DataFrame Indexed by Date ===
            vals
date            
2024-08-05     1
2025-09-15     2
2026-10-25     3


----- Demo Question 3 -----

=== Generated 2-Hourly Time Series (First 5 Rows) ===
                     data
1999-07-21 00:00:00    55
1999-07-21 02:00:00    42
1999-07-21 04:00:00    49
1999-07-21 06:00:00    22
1999-07-21 08:00:00    43


<div style="height: 20px; background: linear-gradient(to bottom, red 0%, red 2px, rgba(255, 0, 0, 0.5) 3px, transparent 100%); border-radius: 6px 6px 0 0; margin: 15px 0;"></div>

### Exercise 1: Parsing Timestamps and Building a Time-Indexed Table

**Goal:** Convert messy date strings and multi-column date records into clean Pandas `DatetimeIndex` tables.

**Requirements:**
1. Parse the custom timestamp string `'15-Nov-2025 | 09:45'` into a Pandas datetime object using the proper `format` codes, and print its `month` and `minute` attributes.
2. Create a DataFrame named `sensor_logs` with columns:
   - `'year'`: `[2025, 2025, 2025]`
   - `'month'`: `[11, 11, 12]`
   - `'day'`: `[10, 20, 1]`
   - `'temp_c'`: `[18.4, 16.2, 14.9]`
3. Combine `'year'`, `'month'`, and `'day'` into a single datetime index, remove the three original date columns, and print the resulting DataFrame.
4. Using `pd.date_range()`, generate a sequence of `7` daily dates starting on `'2026-01-01'` and pair them as the index for a new DataFrame containing `7` random numbers.

<details>
<summary>💡 Need a hint?</summary>

- Match every separator (`-`, `|`, `:`) and character in `'15-Nov-2025 | 09:45'` when writing your format string.
- Remember `%b` matches 3-letter abbreviated month names like `'Nov'`.
- Passing a list of column names `df[['year', 'month', 'day']]` into `pd.to_datetime()` builds dates automatically.
- Use `periods=7` and `freq='D'` inside `pd.date_range()` for step 4.
</details>

1. Parse the custom timestamp string `'15-Nov-2025 | 09:45'` into a Pandas datetime object using the proper `format` codes, and print its `month` and `minute` attributes.

In [16]:
import pandas as pd
import numpy as np


date_str = '15-Nov-2025 | 09:45'

trimmed = pd.to_datetime(date_str, format='%d-%b-%Y | %H:%M')
print('Time:', trimmed)

print('Month:', trimmed.month)

print('Minute:', trimmed.minute)

Time: 2025-11-15 09:45:00
Month: 11
Minute: 45


2. Create a DataFrame named `sensor_logs` with columns:
   - `'year'`: `[2025, 2025, 2025]`
   - `'month'`: `[11, 11, 12]`
   - `'day'`: `[10, 20, 1]`
   - `'temp_c'`: `[18.4, 16.2, 14.9]`

In [36]:
df = pd.DataFrame({
    'year': [2025, 2025, 2025],
    'month': [11, 11, 12],
    'day' : [10, 20, 1],
    'temp_c' : [18.4, 16.2, 14.9]
})

df

,year,month,day,temp_c
0,2025,11,10,18.4
1,2025,11,20,16.2
2,2025,12,1,14.9


3. Combine `'year'`, `'month'`, and `'day'` into a single datetime index, remove the three original date columns, and print the resulting DataFrame.

In [37]:
df['date'] = pd.to_datetime(df[['year', 'month', 'day']])

df

,year,month,day,temp_c,date
0,2025,11,10,18.4,2025-11-10
1,2025,11,20,16.2,2025-11-20
2,2025,12,1,14.9,2025-12-01


In [38]:
df.set_index('date', inplace=True)

df

,year,month,day,temp_c
date,,,,
2025-11-10,2025,11,10,18.4
2025-11-20,2025,11,20,16.2
2025-12-01,2025,12,1,14.9


4. Using `pd.date_range()`, generate a sequence of `7` daily dates starting on `'2026-01-01'` and pair them as the index for a new DataFrame containing `7` random numbers.

In [42]:
date_arr = pd.date_range(start='2026-01-01', periods=7, freq='D')

date_arr

DatetimeIndex(['2026-01-01', '2026-01-02', '2026-01-03', '2026-01-04',
               '2026-01-05', '2026-01-06', '2026-01-07'],
              dtype='datetime64[ns]', freq='D')

In [45]:
df_date_arr = pd.DataFrame({
    'date' : date_arr,
    'values' : np.random.randn(7)
})

df_date_arr

,date,values
0,2026-01-01,0.589057
1,2026-01-02,0.338721
2,2026-01-03,-1.698849
3,2026-01-04,1.457196
4,2026-01-05,1.681092
5,2026-01-06,-0.750906
6,2026-01-07,1.166549


<div style="height: 24px; background: linear-gradient(to top, red 0%, rgba(255, 0, 0, 0.4) 35%, transparent 100%); border-radius: 0 0 8px 8px; margin: 15px 0;"></div>

---

## 4.2 Slicing, Timedelta Math, and Resampling

### Concept Summary
Once your DataFrame uses a `DatetimeIndex`, Pandas unlocks three superpowers:
1. **Smart Date Slicing:** Grab an entire month or day with a short string.
2. **Timedelta Math:** Subtract dates to find exact elapsed durations.
3. **Resampling:** Group rows into time buckets (daily, weekly, monthly) just like `.groupby()`.

---

### Sample DataFrame Used Below
Imagine a 2-hourly sensor dataset running from July to September 1999:
```text
                     data
1999-07-21 00:00:00   0.5
1999-07-21 02:00:00   1.5
1999-07-21 04:00:00   0.2
...
1999-09-21 00:00:00   0.8
```

---

### 1. Smart Date Slicing with `.loc[]`
Pandas supports **partial string indexing**—you only need to type the part of the date you care about:

- **Slice an exact start-to-end window (inclusive):**
  ```python
  df.loc['1999-08-21 08:00:00':'1999-08-22 08:00:00']
  ```
- **Grab an entire month with 7 characters:**
  ```python
  df.loc['1999-09']      # Returns EVERY row in September 1999
  ```
- **Grab all hours of a single day:**
  ```python
  df.loc['1999-09-20']   # Returns all 12 two-hourly rows on Sept 20, 1999
  ```

---

### 2. Date Arithmetic & `Timedelta` Objects
Subtracting two timestamps produces a **`Timedelta`** (exact duration in days, hours, minutes, and seconds):

```python
# Subtract the very first timestamp from every row's timestamp
df['Time_from_start'] = df.index - df.index[0]

# Output sample on Sept 20:
#                      data   Time_from_start
# 1999-09-20 00:00:00  -0.02   61 days 00:00:00
# 1999-09-20 02:00:00  -0.13   61 days 02:00:00
```

---

### 3. Resampling: `.groupby()` for Time

#### 💡 The Simplest Way to Understand Resampling
Think of **`df.resample()`** as **`df.groupby()` for time**.
- With `.groupby('Department')`, Pandas puts rows into buckets by **category name**.
- With `.resample('ME')`, Pandas puts rows into buckets by **calendar time** (every day, every week, or every month) and calculates one summary number per bucket.

```text
=========================================================================================
                  HOW DOWNSAMPLING WORKS (Example: Hourly -> Daily)
=========================================================================================

  [ Raw Data: Many rows per day ]        [ Time Buckets ]        [ Final Summary ]
  2024-01-01 08:00  ->  10 mm
  2024-01-01 14:00  ->  20 mm   ----->   Bucket: Jan 01   --sum()-->  2024-01-01 : 35 mm
  2024-01-01 20:00  ->   5 mm
  ---------------------------------------------------------------------------------------
  2024-01-02 08:00  ->   2 mm   ----->   Bucket: Jan 02   --sum()-->  2024-01-02 : 10 mm
  2024-01-02 14:00  ->   8 mm
=========================================================================================
```

#### Two Types of Resampling
1. **Downsampling (Zooming Out — Most Common):** Many small rows $\rightarrow$ fewer big summary rows (e.g., 2-hourly readings $\rightarrow$ weekly totals or monthly averages).
2. **Upsampling (Zooming In):** Fewer big rows $\rightarrow$ many small rows (e.g., monthly data $\rightarrow$ daily rows, filled using `.ffill()`).

#### Core Syntax & Common Time Rules
```python
df.resample(rule)['Column'].math_function()
```

| Rule Code | Bucket Size | Example Use Case |
| :---: | :--- | :--- |
| `'D'` | **D**aily (1 day per row) | Total daily rainfall or daily website visits |
| `'W'` | **W**eekly (1 week per row) | Weekly total store sales |
| `'ME'` | **M**onth **E**nd (1 month per row) | Average monthly temperature |
| `'YE'` | **Y**ear **E**nd (1 year per row) | Total annual company revenue |

*(Note: Older Pandas versions used `'M'` and `'Y'`; modern Pandas uses `'ME'` and `'YE'`.)*

#### Resampling Examples

- **Example A: Downsampling to Monthly Average (`'ME'`)**
  ```python
  # Groups all 2-hourly readings into monthly buckets and takes the mean
  monthly_mean = df.resample('ME')['data'].mean()
  print(monthly_mean)
  # Output:
  # 1999-07-31    0.008432
  # 1999-08-31    0.047778
  # 1999-09-30   -0.088739
  # Freq: ME, Name: data, dtype: float64
  ```

- **Example B: Downsampling to Weekly Totals (`'W'`)**
  ```python
  # Groups readings into 7-day Sunday-ending weeks and sums them up
  weekly_sum = df.resample('W')['data'].sum()
  print(weekly_sum.head(3))
  # Output:
  # 1999-07-25   -10.377383
  # 1999-08-01    13.679115
  # 1999-08-08     3.479505
  # Freq: W-SUN, Name: data, dtype: float64
  ```

- **Example C: Upsampling Monthly Data to Daily (`'D'`)**
  ```python
  # Expands monthly rows into daily rows and forward-fills empty days
  daily_series = monthly_mean.resample('D').ffill()
  print(daily_series.head(3))
  # Output:
  # 1999-07-31    0.008432
  # 1999-08-01    0.008432
  # 1999-08-02    0.008432
  ```

In [9]:
# --- Demo: Date Slicing, Timedelta & Resampling ---
import pandas as pd
import numpy as np


# Setup 2-hourly synthetic dataset
np.random.seed(42)
dr = pd.date_range(start='1999-07-21', end='1999-09-21', freq='2h')
df = pd.DataFrame({'data': np.random.randn(len(dr))}, index=dr)

print("=== Original Table ===")
print(df.head())



# 1. Slice a specific time window
print("\n\n----- Demo Question 1 -----")

dayslice = df.loc['1999-08-21 08:00:00':'1999-08-22 08:00:00']
print("=== Slice from Aug 21 08:00 to Aug 22 08:00 ===")
print(dayslice.head())



# 2. Partial string slicing (grab all of September)
print("\n\n----- Demo Question 2 -----")

sept_data = df.loc['1999-09']
print("=== Total Rows in September 1999:", len(sept_data))



# 3. Timedelta arithmetic (elapsed time from first timestamp)
print("\n\n----- Demo Question 3 -----")

df['Time_from_start'] = df.index - df.index[0]
print("\n=== Sample Rows on 1999-09-20 with Timedelta ===")
print(df.loc['1999-09-20'].head())



# 4. Resampling (Monthly mean and Weekly sum)
print("\n\n----- Demo Question 4 -----")

print("\n=== Monthly Mean (Downsampling) ===")
print(df.resample('ME')['data'].mean())

print("\n=== Weekly Sum (First 5 Weeks) ===")
print(df.resample('W')['data'].sum().head())

=== Original Table ===
                         data
1999-07-21 00:00:00  0.496714
1999-07-21 02:00:00 -0.138264
1999-07-21 04:00:00  0.647689
1999-07-21 06:00:00  1.523030
1999-07-21 08:00:00 -0.234153


----- Demo Question 1 -----
=== Slice from Aug 21 08:00 to Aug 22 08:00 ===
                         data
1999-08-21 08:00:00  0.872321
1999-08-21 10:00:00  0.183342
1999-08-21 12:00:00  2.189803
1999-08-21 14:00:00 -0.808298
1999-08-21 16:00:00 -0.839722


----- Demo Question 2 -----
=== Total Rows in September 1999: 241


----- Demo Question 3 -----

=== Sample Rows on 1999-09-20 with Timedelta ===
                         data  Time_from_start
1999-09-20 00:00:00  0.408253 61 days 00:00:00
1999-09-20 02:00:00 -1.702584 61 days 02:00:00
1999-09-20 04:00:00  1.029156 61 days 04:00:00
1999-09-20 06:00:00  0.472597 61 days 06:00:00
1999-09-20 08:00:00  0.256030 61 days 08:00:00


----- Demo Question 4 -----

=== Monthly Mean (Downsampling) ===
1999-07-31   -0.077903
1999-08-31    0.042

<div style="height: 20px; background: linear-gradient(to bottom, red 0%, red 2px, rgba(255, 0, 0, 0.5) 3px, transparent 100%); border-radius: 6px 6px 0 0; margin: 15px 0;"></div>

### Exercise 2: Time-Window Filtering, Elapsed Duration & Resampling

**Goal:** Slice hourly energy usage records by date labels, calculate elapsed time using `Timedelta`, and aggregate data into daily and weekly totals.

**Requirements:**
1. Create a `DatetimeIndex` named `hours` running every 1 hour (`'1h'`) from `'2026-03-01'` to `'2026-03-21'`.
2. Create a DataFrame `energy_df` indexed by `hours` with a single column `'kwh'` populated with random floats between `1.0` and `5.0` (using `np.random.uniform(1.0, 5.0, len(hours))`).
3. Using partial string indexing with `.loc[]`, extract and print all hourly readings recorded on `'2026-03-10'`.
4. Add a new column `'Elapsed'` that stores the `Timedelta` difference between each row's timestamp and the very first timestamp in the table.
5. Downsample `energy_df` to compute and display:
   - The **daily average** `'kwh'` usage (first 5 days).
   - The **weekly total sum** of `'kwh'` usage.

<details>
<summary>💡 Need a hint?</summary>

- For step 3, passing `'2026-03-10'` directly into `df.loc[...]` selects all 24 hours of that day.
- For step 4, subtract `df.index[0]` from `df.index`.
- For step 5, use `.resample('D')` for daily buckets and `.resample('W')` for weekly buckets, followed by `['kwh'].mean()` or `['kwh'].sum()`.
</details>

In [ ]:
# --- Exercise 2: Your Code Here ---

<div style="height: 24px; background: linear-gradient(to top, red 0%, rgba(255, 0, 0, 0.4) 35%, transparent 100%); border-radius: 0 0 8px 8px; margin: 15px 0;"></div>